In [20]:
from helpers import CFDCase
import matplotlib.pyplot as plt
import numpy as np
import importlib
import helpers
import xarray as xr


In [ ]:
import re
import pandas as pd
def load_results(filename):
# Read file
    with open(filename, "r") as f:
        lines = f.readlines()
    variable_line = next(line for line in lines if "VARIABLES" in line)
    variable_line = next(line for line in lines if "VARIABLES" in line)
    columns = re.findall(r'"(.*?)"', variable_line)
    # Extract I and J from ZONE line
    zone_line = [line for line in lines if "ZONE" in line][0]
    I = int(re.search(r"I=\s*(\d+)", zone_line).group(1))
    J = int(re.search(r"J=\s*(\d+)", zone_line).group(1))
    
    print(f"Grid size: I={I}, J={J}")
    # Find where numeric data starts
    # Find where numeric data starts
    data_start = 0
    for i, line in enumerate(lines):
        if re.match(r"\s*[-+0-9.]", line):
            data_start = i
            break

    # Load all numeric data at once
    data = np.loadtxt(filename, skiprows=data_start)

    # Reshape: (J, I, n_variables)
    data = data.reshape(J, I, len(columns))

    # Build xarray Dataset
    data_vars = {}
    for k, col in enumerate(columns):
        data_vars[col] = (("y", "x"), data[:, :, k])

    ds = xr.Dataset(
        data_vars,
        coords={
            "x": np.arange(I),
            "y": np.arange(J),
        },
    )
    return ds
    
filename ='../case_data/0_uds2/snaps/snap_000400.dat'
load_results(filename)


Grid size: I=40, J=40


In [ ]:
filename = "case_data/0_quick2/0_result.dat"
filename ='case_data/0_uds2/snaps/snap_000400.dat'
# Reshape into grid
X = df["X"].values.reshape(J, I)
Y = df["Y"].values.reshape(J, I)
U = df["U"].values.reshape(J, I)
V = df["V"].values.reshape(J, I)

velocity_mag = np.sqrt(U**2 + V**2)
# --------- PLOTTING ---------
plt.figure(figsize=(8,6))

# Contour plot of velocity magnitude
contour = plt.contourf(X, Y, velocity_mag, levels=10, cmap="plasma")
plt.colorbar(contour, label="Velocity Magnitude")

# Quiver plot (velocity vectors)
step=2
plt.quiver(
    X[::step, ::step],
    Y[::step, ::step],
    U[::step, ::step],
    V[::step, ::step],
    color="white",
    scale=5
)

plt.xlabel("X")
plt.ylabel("Y")
plt.title("Velocity Field")
plt.gca().set_aspect('equal')

plt.tight_layout()
plt.show()



In [10]:
importlib.reload(helpers)
CFDCase = helpers.CFDCase
case = CFDCase( "../case_data/0_quick2/0_result.dat")


[' VARIABLES = "X", "Y", "U", "V", "P"\n', ' ZONE F=POINT, I=          40 , J=          40\n', '   1.25000002E-02   1.25000002E-02   5.06424585E-05  -5.06604156E-05   1.62332878E-02\n', '   3.75000015E-02   1.25000002E-02   1.23372316E-04  -2.21109531E-05   1.61073357E-02\n', '   6.25000000E-02   1.25000002E-02   9.13317635E-05   5.41053232E-05   1.58217866E-02\n', '   8.75000060E-02   1.25000002E-02  -1.00979560E-04   1.38161427E-04   1.54523365E-02\n', '  0.112500004       1.25000002E-02  -4.52863227E-04   2.13679436E-04   1.50633678E-02\n', '  0.137500003       1.25000002E-02  -9.41764563E-04   2.75181519E-04   1.47010032E-02\n', '  0.162500009       1.25000002E-02  -1.53889670E-03   3.21912463E-04   1.43955406E-02\n', '  0.187500000       1.25000002E-02  -2.21582688E-03   3.54981777E-04   1.41656529E-02\n', '  0.212500006       1.25000002E-02  -2.94688204E-03   3.76039156E-04   1.40221752E-02\n', '  0.237500012       1.25000002E-02  -3.70961241E-03   3.86658125E-04   1.39710605E-02

StopIteration: 

In [ ]:


ds = case.ds

X, Y = np.meshgrid(ds.x.values, ds.y.values)

plt.figure(figsize=(6,5))
plt.quiver(X, Y, ds.U.values, ds.V.values, scale=20)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Velocity Field (Re={ds.attrs['RE']})")
plt.axis("equal")
plt.tight_layout()
plt.show()
